# Hybrid search - vector retrieval and knowledge graph

This query forces the system to traverse multiple ontologies. It must link an environmental parameter (using the Environment Ontology, ENVO) to a specific taxonomic entity (NCBITaxon), and then trace the relationship to specific biological processes (Gene Ontology) like nutrient uptake and photosynthesis

In [33]:
simpleQuery = "What is Zostera Marina?"
complexQuery = """How do varying salinity regimes between
 5 and 40 practical salinity units impact the nitrate and phosphate uptake
  of green algae species like Ulva pertusa, and how might similar environmental
   stressors affect the photosynthetic performance of Zostera marina?"""


In [ ]:
#imports
# Path setup — allows importing from src/
import sys
import json
from pathlib import Path
import chromadb
PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Config: reuses the same settings as the main pipeline
from config import (
    CHROMA_DIR, COLLECTION_NAME, EMBEDDING_MODEL_NAME,
    RERANKER_MODEL, TOP_K_RETRIEVAL, TOP_K_RERANK, NEO4J_URI, NEO4J_USER, CHUNK_DIR
)
# Neo4j
from neo4j import GraphDatabase
# Vector retrieval (LangChain)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder
import pandas as pd
# LLM for entity extraction
from langchain_ollama import ChatOllama
# Generation
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
#model load of gemma4:e2b?
# Loads everything once. models, databases, clients

# Embedding model (same one used for indexing)
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"}
)

# ChromaDB vectorstore
vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model
)
print(f"ChromaDB loaded: {vectorstore._collection.count()} chunks")

# Neo4j
driver = GraphDatabase.driver(
    NEO4J_URI, 
    auth=(NEO4J_USER, os.getenv("NEO4J_PASSWORD"))
)


# test
with driver.session() as session:
    result = session.run("MATCH (n) RETURN count(n) AS total")
    print(f"Neo4j loaded: {result.single()['total']} nodes")

# Reranker
reranker = CrossEncoder(RERANKER_MODEL)

# Local LLM for entity extraction
llm = ChatOllama(model="gemma4:e2b")

# OpenAI for generation
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5717.79it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB loaded: 0 chunks
Neo4j loaded: 95740 nodes


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3155.95it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
# entity extraction - i tihnk i should keep using deepseek for this to stay
#  consistent, I still have money there

# 

In [37]:
#cypher query, i know cypher very little
def expand_from_chunks(chunk_ids, driver, max_triplets=40):#40 is arbitrary for now
    """
    Given chunk IDs from vector retrieval, find entities those chunks mention,
    then traverse one hop outward to discover related facts from other papers.
    
    Returns a list of (subject, predicate, object, confidence) tuples.
    """
    cypher = """
    // Start from the chunks vector search returned
    MATCH (c:Chunk) WHERE c.chunk_id IN $chunk_ids
    
    // Find entities mentioned in those chunks
    MATCH (c)-[:MENTIONS]->(entity)
    
    // One hop outward: any domain relationship from those entities
    MATCH (entity)-[r]->(neighbor)
    WHERE type(r) IN ['FOUND_IN','PRODUCES','STUDIED_WITH',
                      'IDENTIFIED_BY','BELONGS_TO','AFFECTS','CONTAINS']
      AND r.confidence >= 0.7
    
    RETURN DISTINCT
        entity.name  AS subject,
        type(r)      AS predicate,
        neighbor.name AS object,
        r.confidence AS confidence
    ORDER BY r.confidence DESC
    LIMIT $max_triplets
    """
    
    with driver.session() as session:
        result = session.run(cypher, chunk_ids=chunk_ids, max_triplets=max_triplets)
        return [(r["subject"], r["predicate"], r["object"], r["confidence"])
                for r in result]


In [38]:
# Test: does the graph actually return sensible triplets
# for the complex query's entry chunks?
print(entry_docs[0].metadata)
# no chunk meta data, I need to fix this


{'year': '2010', 'title': 'Effect of salinity on growth and nutrient uptake of Ulva pertusa (Chlorophyta) from an eelgrass bed', 'authors': 'Tae Seob Choi, Eun Ju Kang, Kwang Young Kim', 'filename': 'algae-2010-25-1-017.pdf'}


In [39]:
sample_file = next(Path(CHUNK_DIR).glob("*.json"))
with open(sample_file) as f:
    data = json.load(f)
print(type(data), len(data) if hasattr(data, '__len__') else '')
print(data[0] if isinstance(data, list) else list(data.keys())[:5])

<class 'dict'> 9
['filename', 'title', 'authors', 'year', 'strategy']


This walks every key and tells me which one holds the chunks and what shape each chunk has. The question is whether each chunk already carries an index field (in which case it can be used directly when building chunk_id) or whether I derive the index from list position with enumerate()

In [40]:
print("All keys:", list(data.keys()))
print()
for key in data.keys():
    value = data[key]
    if isinstance(value, list):
        print(f"{key}: list of {len(value)} items")
        if value:
            print(f"  first item type: {type(value[0]).__name__}")
            if isinstance(value[0], dict):
                print(f"  first item keys: {list(value[0].keys())}")
            else:
                print(f"  first item preview: {str(value[0])[:200]}")
    else:
        print(f"{key}: {type(value).__name__} = {str(value)[:80]}")

All keys: ['filename', 'title', 'authors', 'year', 'strategy', 'chunk_size', 'chunk_overlap', 'num_chunks', 'chunks']

filename: str = algae-1986-1-1-117.pdf
title: str = The Korean Journal of Phycology Vol. 1, No. 1: 117-134 (1986)
authors: list of 1 items
  first item type: str
  first item preview: Kim, Joo Tae and Jun Chung
year: str = 1986
strategy: str = recursive_character
chunk_size: int = 1000
chunk_overlap: int = 0
num_chunks: int = 30
chunks: list of 30 items
  first item type: dict
  first item keys: ['chunk_id', 'text', 'char_length']


In [41]:
print(data["chunks"][0]["chunk_id"])
print(data["chunks"][1]["chunk_id"])
# so i need to put it in the same format as in neo4j for hybrid retrieval, i messed up

0
1


First, I'm using enumerate(paper["chunks"]) to derive the index from position rather than trusting the integer chunk_id in the JSON, because that integer was what threw me off, Neo4j's ingestion clearly used position too, so this guarantees alignment. Second, I'm flattening authors into a string because Chroma's metadata values must be scalars (str, int, float, bool). it can't store lists. 

In [42]:
texts = []
ids = []
metadatas = []

all_files = sorted(CHUNK_DIR.glob("*.json"))
print(f"Processing {len(all_files)} files...")

for json_path in all_files:
    with open(json_path, "r", encoding="utf-8") as f:
        paper = json.load(f)

    # Strip the .pdf to match Neo4j's chunk_id format
    filename_stem = paper["filename"].replace(".pdf", "")

    for i, chunk in enumerate(paper["chunks"]):
        chunk_id = f"{filename_stem}_chunk_{i:03d}"

        texts.append(chunk["text"])
        ids.append(chunk_id)
        metadatas.append({
        "chunk_id": chunk_id,
        "filename": paper.get("filename", "unknown"),
        "title": paper.get("title") or "Untitled",
        "authors": ", ".join(paper["authors"]) if isinstance(paper.get("authors"), list) else (paper.get("authors") or "Unknown"),
        "year": paper.get("year") or "n.d.",
    })


print(f"Collected {len(texts)} chunks")
print(f"First ID:  {ids[0]}")
print(f"Last ID:   {ids[-1]}")

Processing 868 files...
Collected 31741 chunks
First ID:  algae-1986-1-1-117_chunk_000
Last ID:   algae-2024-39-6-7_chunk_063


In [43]:
# Connect to the persistent Chroma store
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Nukes the existing collection
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"No existing collection to delete (or error): {e}")

# Recreate it fresh, using the same embedding model as before
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
)
print(f"Created fresh collection: {COLLECTION_NAME}")

Deleted existing collection: recursive_100
Created fresh collection: recursive_100


In [44]:
# Quick null check, run  before the expensive embedding
for i, m in enumerate(metadatas):
    for key, val in m.items():
        if val is None:
            print(f"NULL found: chunk {i}, key={key}, file={m.get('filename')}")


In [ ]:
BATCH_SIZE = 5000

for start in range(0, len(texts), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(texts))
    collection.add(
        ids=ids[start:end],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
    )
    print(f"Added {end} / {len(texts)}")

print(f"\nFinal collection count: {collection.count()}")

In [ ]:
# Step 1: get entry chunks via existing vector retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
entry_docs = retriever.invoke(complexQuery)
entry_chunk_ids = [d.metadata["chunk_id"] for d in entry_docs]
print(f"Entry chunks: {entry_chunk_ids}")

# Step 2: expand outward in the graph
triplets = expand_from_chunks(entry_chunk_ids, driver)
print(f"\nRetrieved {len(triplets)} triplets:\n")
for subj, pred, obj, conf in triplets[:15]:
    print(f"  ({subj}) -[{pred} {conf:.2f}]-> ({obj})")

In [ ]:
#connect to neo4j i guess, call with saved query, save as a set of *k* length
query = graphQuery(...)
chunks = query(), 
lcDocs = #convert here

In [ ]:
#part where I pull chunks with vector similarity classically, save as a set of *k* length
# Step 1: Vector retrieval — get entry chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_RETRIEVAL})
entry_docs = retriever.invoke(complexQuery)
entry_chunk_ids = [d.metadata["chunk_id"] for d in entry_docs]
print(f"Vector retrieved {len(entry_docs)} chunks")


In [ ]:
# Step 2: Expand outward from vector-retrieved chunks via the knowledge graph
triplets = expand_from_chunks(entry_chunk_ids, driver)
print(f"Graph returned {len(triplets)} triplets:\n")
for subj, pred, obj, conf in triplets:
    print(f"  ({subj}) -[{pred} {conf:.2f}]-> ({obj})")


In [ ]:
# I propose to merge into a set of 2 k length. but is this grounded in literature?
# rerank to have k best? but now im confused. I thought that graph retrieval is suppsoed to do more than just find a chunk yet again
#^^I was wrong to assume this, after reading more about graph retrieval i came up with something better

# Step 3: Compares what vector retrieval found vs what the graph adds

chunk_df = pd.DataFrame([{"source": "vector", "chunk_id": d.metadata["chunk_id"], 
                          "title": d.metadata.get("title", "")} for d in entry_docs])
triplet_df = pd.DataFrame(triplets, columns=["subject", "predicate", "object", "confidence"])

print("=== Vector chunks ===")
display(chunk_df)
print("\n=== Graph triplets (structured facts the vector search can't find) ===")
display(triplet_df)


In [ ]:
# Step 4: Build context. chunks + graph triplets as structured knowledge
from generation.generate import build_context
context, contexts_list = build_context([(1.0, doc) for doc in entry_docs])

# Appends graph facts as a separate section
triplet_lines = [f"- {subject} {predicate} {object} (confidence: {confidence:.2f})" for subject, predicate, object, confidence in triplets]
enriched_context = context + "\n\nRelated knowledge graph facts:\n" + "\n".join(triplet_lines)


In [ ]:
# second based on knowledge graph retrieval


In [ ]:
# finally, generate an answer based on the reranking combined